# 21 · KV cache quantization and measuring quality loss
Level L4 · Part 3 · core · ~25 min · builds on 20

Time: about 25 min to read and run. No GPU is available on the lab machine. The FP8 cache is simulated on the CPU for the made-up model from lesson 20, and the simulation is the measured result. The one GPU step is gated (`needs-gpu`) and prints its commands only.

## Goal
By the end of this lesson you can measure the quality cost of an FP8 KV cache with a 50-prompt regression set. You can report the pass rate with its 95% confidence interval, and say why a perplexity number alone does not settle the question.

## Why it matters
- **Without it:** the KV cache limits how many requests run at once (lesson 05). FP8 halves its size, so the gain looks free. Teams ship it on a perplexity number or a benchmark score, and a support answer that flips from "yes" to "no" goes unnoticed.
- **With it:** half the bytes, twice the tokens. Here is the 70B model with a 20 GiB cache budget, all GPUs together (the arithmetic is in step 2). A regression set with saved golden outputs shows which answers changed and how many.
- **When you don't need it:** if the cache is not the limit, keep FP16 KV. Check the budget first (lesson 05), then pay for the quality check.

## The idea
The KV cache keeps one key and one value vector for every token in every layer. FP8 stores each number in 1 byte instead of 2, so the same memory holds twice the tokens. The cost is rounding: each key and value moves to the nearest FP8 value, and attention reads slightly different numbers. vLLM's scale is 1.0 unless you calibrate, a second place where quality can move. Perplexity averages over every token, so it can stay flat while one answer changes. The check is a fixed set of prompts, a golden output for each, and a pass rate with an interval. The 50 prompts and their golden outputs are made up for this lesson, and the golden outputs come from the FP16 model. So the score measures change from FP16, not correctness. It is like checking a photocopy page by page, not by its average darkness.

## Picture
```mermaid
flowchart LR
  P["50 fixed prompts"] --> G["FP16 KV: golden outputs, saved"]
  P --> Q["FP8 KV: same prompts, greedy decoding"]
  G --> C["compare: same 8 new tokens?"]
  Q --> C
  C --> R["pass rate with 95% interval, perplexity ratio"]
  R --> D{"inside the agreed tolerance?"}
```

## Step by step
### Step 1: the made-up model and the 50 prompts
The model is the made-up 62M-parameter Llama-shaped model from lesson 20 (`../data/20/fp16`, seed 0). It has no tokenizer, so prompts are token ids. The policy is a **made-up stand-in**: the course has no copy of the real Brightside policy, so 80 policy ids come from a fixed seed (the same approach as lesson 18). Each of the 50 questions adds 16 more made-up ids.

In [1]:
import json, math, random
from pathlib import Path

import torch
from transformers import LlamaForCausalLM

torch.manual_seed(21)
OUT = Path("../data/21")
OUT.mkdir(parents=True, exist_ok=True)
rng = random.Random(21)                                     # fixed seed: the same prompts every run
POLICY_IDS = [rng.randrange(5, 16000) for _ in range(80)]   # made-up policy stand-in
PROMPTS = [POLICY_IDS + [rng.randrange(5, 16000) for _ in range(16)] for _ in range(50)]
model = LlamaForCausalLM.from_pretrained("../data/20/fp16", dtype=torch.float32).eval()
print(f"{len(PROMPTS)} prompts of {len(PROMPTS[0])} ids; model {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("prompt 1, first 8 ids:", PROMPTS[0][:8])

./.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 938.07it/s]

50 prompts of 96 ids; model 62.3M parameters
prompt 1, first 8 ids: [2707, 6853, 11306, 6854, 10408, 4613, 7854, 13821]


**What you should see:** `50 prompts of 96 ids` and the parameter count, about 62.3M. The first ids are the same for every prompt: that is the shared policy prefix.

### Step 2: the arithmetic, before any run
Each token stores keys and values in every layer: 2 × 80 layers × 8 KV heads × 128 values per head for Llama 3.1 70B (lesson 03). FP16 uses 2 bytes per value and FP8 uses 1.

In [2]:
GIB, KIB = 2**30, 2**10
BUDGET = 20 * GIB                                            # a 20 GiB cache budget, all GPUs together

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value):
    return 2 * layers * kv_heads * head_dim * bytes_per_value    # 2 = keys and values

for name, width in [("FP16 KV", 2), ("FP8 KV", 1)]:
    per_token = kv_bytes_per_token(80, 8, 128, width)
    print(f"70B {name}: {per_token / KIB:.0f} KiB per token, {BUDGET // per_token:,} tokens in 20 GiB")

70B FP16 KV: 320 KiB per token, 65,536 tokens in 20 GiB
70B FP8 KV: 160 KiB per token, 131,072 tokens in 20 GiB


**What you should see:** 320 KiB per token in FP16 and 160 KiB in FP8, so the same budget holds 65,536 tokens or 131,072. This is only capacity. It says nothing about the answers.

### Step 3: the FP8 rounding, as a function
The GPU engine stores the cache in FP8 itself. On this CPU the same rounding is simulated. Each key and value is rounded to FP8 (format e4m3, the one vLLM's `fp8` setting uses), clipped to ±448, the largest e4m3 value. The scale is 1.0, vLLM's default without calibration. A forward hook on every `k_proj` and `v_proj` applies the rounding, so generation reads the rounded values from its cache. The hook rounds the projection output before the position rotation (RoPE) is applied to keys, so the simulation is close to the GPU kernel, not identical.

In [3]:
FP8_MAX = 448.0

def fp8_roundtrip(x):
    """Store x as FP8 (e4m3) and read it back: the rounding an FP8 cache causes."""
    return x.clamp(-FP8_MAX, FP8_MAX).to(torch.float8_e4m3fn).to(x.dtype)

def simulate_fp8_kv(model):
    """Round every key and value the attention layers write to the cache. Returns the hooks.
    A forward hook's return value replaces the layer output, so the rounded tensor is what attention reads."""
    return [m.register_forward_hook(lambda mod, inp, out: fp8_roundtrip(out))
            for n, m in model.named_modules() if n.endswith(("k_proj", "v_proj"))]

print("round trip of 0.1, 3.3, 500:", fp8_roundtrip(torch.tensor([0.1, 3.3, 500.0])).tolist())

round trip of 0.1, 3.3, 500: [0.1015625, 3.25, 448.0]


**What you should see:** `[0.1015625, 3.25, 448.0]`. Small values lose a little, and 500 is clipped to 448. These are the rounding errors the attention layers will read.

### Step 4: golden outputs and the FP8 outputs
The FP16 model, with no hooks, generates 8 greedy tokens for each prompt (the top token at each step, so the run repeats). Those are the **golden outputs**; they are saved to `../data/21/golden.json`. Then the same prompts run with the FP8 simulation, and the hooks are removed at once.

In [4]:
N_NEW = 8

@torch.no_grad()
def generate(ids):
    out = model.generate(torch.tensor([ids]), max_new_tokens=N_NEW, min_new_tokens=N_NEW,
                         do_sample=False, pad_token_id=0)
    return out[0, len(ids):].tolist()

golden = [generate(p) for p in PROMPTS]
(OUT / "golden.json").write_text(json.dumps(golden))
hooks = simulate_fp8_kv(model)
fp8 = [generate(p) for p in PROMPTS]
for h in hooks:
    h.remove()                                              # later cells use FP16 again
print("golden, prompt 1:", golden[0])
print("FP8,    prompt 1:", fp8[0])

golden, prompt 1: [5228, 14609, 15778, 15344, 15778, 15344, 12435, 15344]
FP8,    prompt 1: [5228, 14609, 15778, 15344, 15778, 15344, 14372, 12435]


**What you should see:** two lists of 8 ids. Prompt 1 matches for six tokens and changes at position 6. The next step counts how many prompts change. Only the first changed position of each answer is counted later; anything after it is not.

### Step 5: the regression score and the perplexity
A prompt **passes** when its 8 new tokens match the golden output exactly. The pass rate comes with a 95% Wilson interval. With 50 prompts the interval is wide, so the set is a tripwire for large losses, not proof of small ones. Perplexity is also computed on the golden tokens, FP16 and FP8, so you can see what it misses.

Two helpers, defined in one cell. `wilson` gives the pass rate and its 95% interval. `golden_nll` gives the summed loss (negative log-probability) of the golden tokens. Perplexity is exp(total NLL / number of tokens); cell 23 does that division.

In [5]:
def wilson(passes, n, z=1.96):
    """Pass rate and its 95% Wilson interval (it stays inside 0 to 1 at 0% and 100%)."""
    p = passes / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return p, centre - half, centre + half

@torch.no_grad()
def golden_nll():
    """Sum of -log p(golden token) over all prompts, teacher-forced on the golden tokens."""
    total = 0.0
    for p, g in zip(PROMPTS, golden):
        logits = model(torch.tensor([p + g])).logits[0, len(p) - 1:-1]
        total += torch.nn.functional.cross_entropy(logits, torch.tensor(g), reduction="sum").item()
    return total

print("wilson(28, 50) gives pass rate, low, high:", [round(x, 3) for x in wilson(28, 50)])

wilson(28, 50) gives pass rate, low, high: [0.56, 0.423, 0.688]


Now the run: the pass count, the interval and the two perplexities.

In [6]:
n_tokens = len(PROMPTS) * N_NEW
passes = sum(g == f for g, f in zip(golden, fp8))
p, lo, hi = wilson(passes, len(PROMPTS))
ppl16 = math.exp(golden_nll() / n_tokens)
hooks = simulate_fp8_kv(model)
ppl8 = math.exp(golden_nll() / n_tokens)
for h in hooks:
    h.remove()
print(f"FP8 KV: {passes} of 50 prompts identical ({p:.0%}, 95% CI {lo:.0%} to {hi:.0%})")
print(f"perplexity on golden tokens: FP16 {ppl16:.3f}, FP8 {ppl8:.3f}, ratio {ppl8 / ppl16:.3f}")

FP8 KV: 28 of 50 prompts identical (56%, 95% CI 42% to 69%)
perplexity on golden tokens: FP16 2004.353, FP8 2020.261, ratio 1.008


**What you should see:** `28 of 50 prompts identical (56%, 95% CI 42% to 69%)` and a perplexity ratio of 1.008. Perplexity barely moves while 22 answers change. That gap is why perplexity alone is not the check. A random model has almost flat logits (perplexity about 2000 here), so top-1 flips are much easier than on a trained model. A random model gives a huge perplexity, so compare the ratio, not the absolute number. The 56% pass rate and the 1.008 ratio show the method only. They say nothing about real FP8 KV quality.

### Step 6: where the answers change
A changed answer is more useful when you know its first changed position. Position 0 is the first new token, the one that follows the prompt directly. Only that first changed position is counted here; later changes in the same answer are not counted.

In [7]:
# first changed position only: the first j where the golden and FP8 tokens differ
changed = [next(j for j in range(N_NEW) if g[j] != f[j]) for g, f in zip(golden, fp8) if g != f]
print(f"{len(changed)} prompts changed; first changed positions:", sorted(changed))

22 prompts changed; first changed positions: [0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 5, 6, 6, 6, 6, 6]


**What you should see:** `22 prompts changed`, with first changed positions from 0 to 6. Two changes are at position 0, where the first token itself moved.

### Step 7: the same check on a GPU (not run here)
On a GPU, vLLM stores the cache in FP8 itself. The pinned vLLM image (v0.32.0) lists `fp8` KV types. The check is the same: serve one checkpoint twice, send the 50 prompts at temperature 0, and compare with `golden.json`. No GPU run is recorded here, so the cell below only prints commands.

In [8]:
print("FP16 KV:  vllm serve <checkpoint> --kv-cache-dtype auto")
print("FP8 KV:   vllm serve <checkpoint> --kv-cache-dtype fp8")
print("skipped: no GPU run is recorded; the CPU simulation above is the measured result")

⏭  Skipped: this cell needs [gpu,slow]. Run it yourself, or re-run with TUT_ENABLE=gpu,slow.


## At work
- **Gates before traffic:** teams check a new KV setting on an offline golden set, then canary it. Sign-off uses hundreds of items. → lesson 35 (rollouts with eval gates).
- **Bit width is not the whole story:** Kurtic et al. (ACL 2025, arXiv 2411.02355) found FP8 weight-and-activation quantization effectively lossless on Llama 3.1. It did not test the KV cache alone, so measure yours. → lesson 20.
- **Hardware decides the speed:** FP8 KV saves memory on any card, but attention is only faster on GPUs with FP8 math. Check your card's FP8 support before you expect a speedup. → lesson 24.
- **Go deeper:** calibrated KV scales come from llm-compressor's `kv_cache_scheme` (checked in 0.14.0), and `--kv-cache-dtype-skip-layers` keeps sensitive layers in FP16.

Sources: vLLM quantized KV cache, https://docs.vllm.ai/ · Kurtic et al., https://arxiv.org/abs/2411.02355 · llm-compressor, https://github.com/vllm-project/llm-compressor

## Common mistakes
**1. Doing FP8 math on an FP8 tensor.** The rounded values are stored as FP8, but the arithmetic must use a normal dtype. Adding two FP8 tensors fails.

In [9]:
a = torch.tensor([1.0]).to(torch.float8_e4m3fn)
a + a

NotImplementedError: "add_stub" not implemented for 'Float8_e4m3fn'

The fix: round with `fp8_roundtrip`, which returns the original dtype, as the hook does.

**2. Leaving the hooks on.** If the FP8 hooks stay on, the next "golden" run reads FP8 values and the check passes for free. Remove them in the cell that installs them, as steps 4 and 5 do.

**3. Reading 50 prompts as exact.** A drop of 4 prompts is 8 points (4 / 50), and the 95% interval at 50 prompts is about ±13 points. So 8 points is within noise. The Wilson interval gives this uncertainty and stays inside 0 to 100%. Repeat with more prompts before you decide.

## Try it
Using `golden` and `fp8` from step 4, how many prompts change at position 0 (the first new token), and how many change only later? Write the two counts in the cell below.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
first = [next(j for j in range(N_NEW) if g[j] != f[j]) for g, f in zip(golden, fp8) if g != f]
print("changed at position 0:", first.count(0), "| changed later:", len(first) - first.count(0))

changed at position 0: 2 | changed later: 20


## Recap
- An FP8 KV cache halves the bytes per token, so the same budget holds twice the tokens. It changes the numbers attention reads, so answers can flip.
- Measure with golden outputs from the FP16 model on a fixed prompt set. The prompts and golden outputs here are made up, so the score measures change from FP16, not correctness. Report the pass rate and its interval.
- Perplexity can stay close to 1 (as in this toy run) while answers flip. Use it as a second signal, not the decision.
- A 50-prompt set is a tripwire. Escalate to hundreds of prompts before sign-off.

**Interview:** "Turning on FP8 KV dropped your score by 2 points. Is that real?" Strong answer: with 50 prompts the 95% interval is about ±13 points, so 2 points is noise. I would compare the runs on the same prompts, look at where answers changed, run hundreds of prompts, and check long-context cases.